# Gradient Cobra

## Combine Classifier

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(
    f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features\n"
    f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features"
)

Training set: 455 samples, 30 features
Test set: 114 samples, 30 features


In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from cobra.combine_classifier import CombineClassifier

model = CombineClassifier(
    splitter="holdout",
    distance="hamming",
    kernel="indicator",
    aggregator="majority_vote",
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

ImportError: cannot import name 'BaseSpaceProjector' from 'cobra.core.spaces.base' (/Users/ougi/Documents/Project/kfc-procedure/src/cobra/core/spaces/base.py)

In [ ]:
preds = model.predict(X_test)
accuracy = (preds == y_test).mean()
print(f"Test set accuracy: {accuracy:.4f}")

Test set accuracy: 0.8596


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

baselines = {
    "decision_tree": DecisionTreeClassifier(random_state=42),
    "logistic_regression": LogisticRegression(max_iter=5000),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "gradient_boosting": GradientBoostingClassifier(),
    "knn": KNeighborsClassifier(),
    "svm": SVC()
}

results = []

for name, model in baselines.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    results.append(("baseline_" + name, acc))

model = CombineClassifier(
    estimators=baselines.keys(),
    splitter="holdout",
    distance="hamming",
    kernel="indicator",
    aggregator="majority_vote",
    random_state=42
)
model.fit(X_train, y_train)
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
results.append(("combine_classifier", acc))

for name, acc in results:
    print(f"{name}: {acc:.4f}")


baseline_decision_tree: 0.9474
baseline_logistic_regression: 0.9561
baseline_random_forest: 0.9649
baseline_gradient_boosting: 0.9561
baseline_knn: 0.9561
baseline_svm: 0.9474
combine_classifier: 0.9737


In [ ]:
from cobra.core.estimators.base import BaseEstimator, EstimatorFactory
EstimatorFactory.available()

['decision_tree',
 'dummy_mean',
 'gradient_boosting',
 'knn',
 'lasso',
 'linear',
 'logistic_regression',
 'mean_regressor',
 'random_forest',
 'ridge',
 'svm']

# GradientCobra

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
X, y = fetch_california_housing(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(
    f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features\n"
    f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features"
)

Training set: 16512 samples, 8 features
Test set: 4128 samples, 8 features


In [2]:
from cobra.gradientcobra import GradientCOBRA
import numpy as np

model = GradientCOBRA(
    distance="euclidean",
    kernel="rbf",
    aggregator="weighted_mean",
    optimizer="gradient_descent",
    random_state=42
)
model.fit(X_train, y_train)

,estimators,None
,estimators_params,None
,distance,'euclidean'
,distance_params,None
,kernel,'rbf'
,kernel_params,None
,aggregator,'weighted_mean'
,aggregator_params,None
,loss,'mse'
,loss_params,None
,optimizer,'gradient_descent'


In [4]:
model.optimization_outputs_

{'method': 'grad',
 'params': array([0.35300666]),
 'history': [array([0.49431805]),
  array([0.49752971]),
  array([0.50078113]),
  array([0.50407281]),
  array([0.50740512]),
  array([0.51077844]),
  array([0.51419318]),
  array([0.51764966]),
  array([0.5211483]),
  array([0.52468931]),
  array([0.52827305]),
  array([0.53189961]),
  array([0.53556932]),
  array([0.53928224]),
  array([0.54303846]),
  array([0.54683806]),
  array([0.55068099]),
  array([0.55456717]),
  array([0.55849636]),
  array([0.56246832]),
  array([0.56648273]),
  array([0.57053904]),
  array([0.57463672]),
  array([0.57877507]),
  array([0.58295322]),
  array([0.58717013]),
  array([0.59142473]),
  array([0.59571559]),
  array([0.60004113]),
  array([0.60439971]),
  array([0.60878926]),
  array([0.61320762]),
  array([0.61765217]),
  array([0.6221202]),
  array([0.62660854]),
  array([0.63111378]),
  array([0.63563212]),
  array([0.64015937]),
  array([0.64469088]),
  array([0.64922172]),
  array([0.65374632]

In [3]:
import numpy as np
preds = model.predict(X_test)
mse = np.mean((preds - y_test) ** 2)
print(f"Test set MSE: {mse:.4f}")

Test set MSE: 0.7670


In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

baselines = {
    "linear": LinearRegression(),
    "ridge": Ridge(),
    "random_forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "svm": SVR()
}
results = []
for name, model in baselines.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mse = mean_squared_error(y_test, preds)
    results.append(("baseline_" + name, mse))

model = GradientCOBRA(
    estimators=baselines.keys(),
    distance="euclidean",
    kernel="rbf",
    aggregator="weighted_mean",
    random_state=42
)
model.fit(X_train, y_train)
preds = model.predict(X_test)
mse = mean_squared_error(y_test, preds)
results.append(("gradient_cobra", mse))

for name, mse in results:
    print(f"{name}: {mse:.4f}")

Gradient Descent: 100%|██████████| 10/10 [00:18<00:00,  1.81s/it, best=0.7061, grad=0.7164, score=0.7061]


baseline_linear: 0.5559
baseline_ridge: 0.5558
baseline_random_forest: 0.2539
baseline_svm: 1.3320
gradient_cobra: 0.7097


# MixCobra

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
X, y = fetch_california_housing(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(
    f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features\n"
    f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features"
)

Training set: 16512 samples, 8 features
Test set: 4128 samples, 8 features


In [ ]:
from cobra.mixcobra import MixCOBRARegressor
model = MixCOBRARegressor(
    splitter="kfold",
    distance="euclidean",
    kernel="rbf",
    aggregator="weighted_mean",
    loss="mse",
    optimizer="grad",
    random_state=42
)
model.fit(X_train, y_train)

Gradient Descent: 100%|██████████| 10/10 [00:54<00:00,  5.43s/it, best=1.1527, grad=0.2195, score=1.1527]


,estimators,None
,estimators_params,None
,splitter,'kfold'
,splitter_params,None
,distance,'euclidean'
,distance_params,None
,kernel,'rbf'
,kernel_params,None
,aggregator,'weighted_mean'
,aggregator_params,None
,loss,'mse'


In [ ]:
model.optimization_outputs_

{'alpha': np.float64(0.48410493855949266),
 'beta': np.float64(0.5155506267163208),
 'risk': 1.1527263564416708,
 'histories': [{'iteration': 0,
   'x': array([0.4983623 , 0.50154702]),
   'score': 1.157158173760247,
   'best_score': 1.157158173760247,
   'gradient': array([ 0.16377022, -0.15470152])},
  {'iteration': 1,
   'x': array([0.49673534, 0.50309585]),
   'score': 1.1566543182808822,
   'best_score': 1.1566543182808822,
   'gradient': array([ 0.16269542, -0.15488372])},
  {'iteration': 2,
   'x': array([0.49511912, 0.5046465 ]),
   'score': 1.1561533776012887,
   'best_score': 1.1561533776012887,
   'gradient': array([ 0.16162191, -0.15506466])},
  {'iteration': 3,
   'x': array([0.49351363, 0.50619894]),
   'score': 1.1556553277799706,
   'best_score': 1.1556553277799706,
   'gradient': array([ 0.16054965, -0.15524435])},
  {'iteration': 4,
   'x': array([0.49191884, 0.50775317]),
   'score': 1.1551601445429363,
   'best_score': 1.1551601445429363,
   'gradient': array([ 0.15

In [ ]:
from sklearn.metrics import mean_squared_error
preds = model.predict(X_test, pred_X=X_test)
mse = mean_squared_error(y_test, preds)
print(f"Test set MSE: {mse:.4f}")

Test set MSE: 1.0975


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error

from cobra.mixcobra import MixCOBRARegressor

# ======================
# SINGLE MODELS
# ======================
single_models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
    "SVR": SVR(C=5.0, epsilon=0.1)
}

results = {}

# train single models
for name, model in single_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = mean_squared_error(y_test, pred)

# ======================
# MIXCOBRA
# ======================
mix = MixCOBRARegressor(
    kernel="rbf",
    distance="euclidean",
    aggregator="weighted_mean",
    random_state=42
)

mix.fit(X_train, y_train)
mix_pred = mix.predict(X_test, pred_X=X_test)

results["MixCOBRA"] = mean_squared_error(y_test, mix_pred)

# ======================
# RESULTS
# ======================
print("\n===== MSE COMPARISON =====\n")

for name, score in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:20s} : {score:.4f}")

Gradient Descent: 100%|██████████| 10/10 [00:53<00:00,  5.37s/it, best=1.1524, grad=0.2200, score=1.1524]



===== MSE COMPARISON =====

RandomForest         : 0.2537
Ridge                : 0.5558
LinearRegression     : 0.5559
MixCOBRA             : 1.0975
SVR                  : 1.2197
